# Data Profiling — 4 Approved Data Sources
**MedTrack_DV — Milestone 1, Step: Data Profiling & Dataset Validation**

Before any cleaning or merging, profile all 4 approved sources:
1. **HMIS** (core relational dataset — 19 tables)
2. **Beds Management** (resource dataset — 4 tables)
3. **Readmission** (patient/outcome dataset — 1 file)
4. **Inpatient Discharges / SPARCS** (large inpatient dataset — sample only)

**Goal:** understand shape, columns, missing values, duplicates, and identify common keys (`hospital_id`, `patient_id`, `admission_id`, `department_id`) BEFORE deciding how the 4 final grain-based tables will be built.

This notebook does NOT clean or merge anything — profiling only, per mentor's rule: *"If you cannot explain what one row represents, do not proceed to the next stage."*

In [1]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW_DIR = "../data/raw"

HMIS_DIR = f"{RAW_DIR}/hmis"
BEDS_DIR = f"{RAW_DIR}/beds_management"
READMIT_DIR = f"{RAW_DIR}/readmission"
INPATIENT_DIR = f"{RAW_DIR}/inpatient_discharges"

## Helper: profile any dataframe
Same function reused across all 4 sources so results are comparable.

In [2]:
def profile(df, name):
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print("Shape:", df.shape)
    print("\nColumns:", list(df.columns))
    print("\nDtypes:\n", df.dtypes)
    print("\nMissing values (non-zero only):")
    miss = df.isnull().sum()
    print(miss[miss > 0] if (miss > 0).any() else "None")
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\nSample:")
    return df.head(3)

## 1. HMIS — Core Dataset (19 tables)
List all files first, then profile each one.

In [3]:
hmis_files = [f for f in os.listdir(HMIS_DIR) if f.endswith(('.csv', '.xlsx'))]
print(f"Found {len(hmis_files)} HMIS files:")
for f in sorted(hmis_files):
    print(" -", f)

Found 19 HMIS files:
 - admission.csv
 - bed.csv
 - billing.csv
 - billing_detail.csv
 - department.csv
 - diagnostic_test.csv
 - disease.csv
 - doctor.csv
 - drug.csv
 - drug_inventory.csv
 - drug_manufacturer.csv
 - employee.csv
 - insurance_provider.csv
 - patient.csv
 - patient_diagnostic.csv
 - patient_insurance.csv
 - prescription.csv
 - staff_assignment.csv
 - ward.csv


In [4]:
hmis_tables = {}
for f in hmis_files:
    name = os.path.splitext(f)[0]
    path = f"{HMIS_DIR}/{f}"
    try:
        if f.endswith('.csv'):
            hmis_tables[name] = pd.read_csv(path)
        else:
            hmis_tables[name] = pd.read_excel(path)
    except Exception as e:
        print(f"Failed to load {f}: {e}")

print(f"Loaded {len(hmis_tables)} HMIS tables")

Loaded 19 HMIS tables


In [5]:
for name, df in hmis_tables.items():
    profile(df, f"HMIS: {name}")


HMIS: admission
Shape: (45000, 10)

Columns: ['admission_id', 'admission_date', 'discharge_date', 'admission_type', 'admission_status', 'patient_id', 'department_id', 'ward_id', 'bed_id', 'disease_id']

Dtypes:
 admission_id         int64
admission_date      object
discharge_date      object
admission_type      object
admission_status    object
patient_id           int64
department_id        int64
ward_id              int64
bed_id               int64
disease_id           int64
dtype: object

Missing values (non-zero only):
None

Duplicate rows: 0

Sample:

HMIS: bed
Shape: (415, 4)

Columns: ['bed_id', 'bed_number', 'bed_status', 'ward_id']

Dtypes:
 bed_id         int64
bed_number    object
bed_status    object
ward_id        int64
dtype: object

Missing values (non-zero only):
None

Duplicate rows: 0

Sample:

HMIS: billing
Shape: (45000, 8)

Columns: ['bill_id', 'bill_date', 'total_amount', 'insurance_covered_amount', 'patient_payable_amount', 'payment_status', 'payment_mode', 'adm

### HMIS key columns check
Identify which tables carry `hospital_id`, `patient_id`, `admission_id`, `department_id`, `ward_id`, `bed_id`, `doctor_id`, `staff_id` — these are the linking keys the mentor's document requires.

In [6]:
key_candidates = ['hospital_id', 'patient_id', 'admission_id', 'department_id',
                   'ward_id', 'bed_id', 'doctor_id', 'staff_id', 'employee_id']

print(f"{'Table':<25}", " | ".join(k.ljust(15) for k in key_candidates))
for name, df in hmis_tables.items():
    cols_lower = [c.lower() for c in df.columns]
    marks = [("YES" if k in cols_lower else "-").ljust(15) for k in key_candidates]
    print(f"{name:<25}", " | ".join(marks))

Table                     hospital_id     | patient_id      | admission_id    | department_id   | ward_id         | bed_id          | doctor_id       | staff_id        | employee_id    
admission                 -               | YES             | YES             | YES             | YES             | YES             | -               | -               | -              
bed                       -               | -               | -               | -               | YES             | YES             | -               | -               | -              
billing                   -               | -               | YES             | -               | -               | -               | -               | -               | -              
billing_detail            -               | -               | -               | -               | -               | -               | -               | -               | -              
department                -               | -               | -       

## 2. Beds Management — Resource Dataset (4 tables)

In [7]:
beds_patients = pd.read_csv(f"{BEDS_DIR}/patients.csv")
beds_staff = pd.read_csv(f"{BEDS_DIR}/staff.csv")
beds_schedule = pd.read_csv(f"{BEDS_DIR}/staff_schedule.csv")
beds_services = pd.read_csv(f"{BEDS_DIR}/services_weekly.csv")

profile(beds_patients, "Beds Management: patients")


Beds Management: patients
Shape: (1000, 7)

Columns: ['patient_id', 'name', 'age', 'arrival_date', 'departure_date', 'service', 'satisfaction']

Dtypes:
 patient_id        object
name              object
age                int64
arrival_date      object
departure_date    object
service           object
satisfaction       int64
dtype: object

Missing values (non-zero only):
None

Duplicate rows: 0

Sample:


,patient_id,name,age,arrival_date,departure_date,service,satisfaction
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83


In [8]:
profile(beds_staff, "Beds Management: staff")


Beds Management: staff
Shape: (110, 4)

Columns: ['staff_id', 'staff_name', 'role', 'service']

Dtypes:
 staff_id      object
staff_name    object
role          object
service       object
dtype: object

Missing values (non-zero only):
None

Duplicate rows: 0

Sample:


,staff_id,staff_name,role,service
0,STF-5ca26577,Allison Hill,doctor,emergency
1,STF-02ae59ca,Noah Rhodes,doctor,emergency
2,STF-d8006e7c,Angie Henderson,doctor,emergency


In [9]:
profile(beds_schedule, "Beds Management: staff_schedule")


Beds Management: staff_schedule
Shape: (6552, 6)

Columns: ['week', 'staff_id', 'staff_name', 'role', 'service', 'present']

Dtypes:
 week           int64
staff_id      object
staff_name    object
role          object
service       object
present        int64
dtype: object

Missing values (non-zero only):
None

Duplicate rows: 0

Sample:


,week,staff_id,staff_name,role,service,present
0,1,STF-b77cdc60,Allison Hill,doctor,emergency,1
1,2,STF-b77cdc60,Allison Hill,doctor,emergency,1
2,3,STF-b77cdc60,Allison Hill,doctor,emergency,0


In [10]:
profile(beds_services, "Beds Management: services_weekly")


Beds Management: services_weekly
Shape: (208, 10)

Columns: ['week', 'month', 'service', 'available_beds', 'patients_request', 'patients_admitted', 'patients_refused', 'patient_satisfaction', 'staff_morale', 'event']

Dtypes:
 week                     int64
month                    int64
service                 object
available_beds           int64
patients_request         int64
patients_admitted        int64
patients_refused         int64
patient_satisfaction     int64
staff_morale             int64
event                   object
dtype: object

Missing values (non-zero only):
None

Duplicate rows: 0

Sample:


,week,month,service,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,event
0,1,1,emergency,32,76,32,44,67,70,none
1,1,1,surgery,45,130,45,85,83,78,flu
2,1,1,general_medicine,37,201,37,164,97,43,flu


**Note:** This dataset has NO `hospital_id` (single hospital, no multi-hospital field) and NO `admission_id`/`patient_id`-level readmission flag. It links internally only via `service` (department-equivalent) and `staff_id`/`week`. It is being used strictly for the **Resource Utilization** dashboard as the mentor specified — not merged directly with HMIS admission-level data.

## 3. Readmission Dataset — Patient/Outcome Dataset

In [11]:
readmit_files = [f for f in os.listdir(READMIT_DIR) if f.endswith(('.csv', '.xlsx'))]
print("Readmission files found:", readmit_files)

Readmission files found: ['Healthcare Data Analysis for readmission.csv']


In [12]:
readmit_file = readmit_files[0]
readmit_path = f"{READMIT_DIR}/{readmit_file}"

if readmit_file.endswith('.csv'):
    readmission = pd.read_csv(readmit_path)
else:
    readmission = pd.read_excel(readmit_path)

profile(readmission, "Readmission Dataset")


Readmission Dataset
Shape: (10000, 26)

Columns: ['hospital_name', 'Admission_date', 'hospital_id', 'hospital_beds_available', 'occupied_beds', 'hospital_ward', 'patient_id', 'patient_gender', 'patient_age', 'patient_race', 'patient_sat_score', 'patient_first_initial', 'patient_last_name', 'patient_waittime', 'department_referral', 'time_slot', 'doctor_id', 'doctor_name', 'doctor_specialty', 'patient_assigned_doctor', 'patient_checkin_date', 'patient_checkout_date', 'patient_disease', 'patient_length_of_stay', 'discharge_status', 'readmission']

Dtypes:
 hospital_name              object
Admission_date             object
hospital_id                 int64
hospital_beds_available     int64
occupied_beds               int64
hospital_ward              object
patient_id                  int64
patient_gender             object
patient_age                 int64
patient_race               object
patient_sat_score           int64
patient_first_initial      object
patient_last_name          obj

,hospital_name,Admission_date,hospital_id,hospital_beds_available,occupied_beds,hospital_ward,patient_id,patient_gender,patient_age,patient_race,patient_sat_score,patient_first_initial,patient_last_name,patient_waittime,department_referral,time_slot,doctor_id,doctor_name,doctor_specialty,patient_assigned_doctor,patient_checkin_date,patient_checkout_date,patient_disease,patient_length_of_stay,discharge_status,readmission
0,The Johns Hopkins Hospital,30-01-2024,3946,260,90,Maternity,1421,Female,48,Asian,490,t,Rogers,120,Urology or Nephrology,02:34:00 PM,1725,Justin Morris,Rheumatology,False,14-07-2024,19-07-2024,Anxiety Disorders,6,Deceased,1
1,The Johns Hopkins Hospital,20-03-2022,7147,250,90,ICU,4922,Male,40,Black,1210,i,Spencer,60,Neurology,04:07:00 PM,7510,Latoya Moss,Pulmonology or Allergy and Immunology,False,19-07-2024,07/07/2024,Urinary Tract Infection (UTI),2,Deceased,1
2,The Johns Hopkins Hospital,13-07-2021,4174,350,220,Pediatrics,9804,Female,74,Black,920,Y,Hall,120,Neurology,07:16:00 PM,7137,Michael Allen,Neurology,False,20-07-2024,29-06-2024,Hypertension (High Blood Pressure),17,Recovered,0


## 4. Inpatient Discharges (SPARCS) — Large Dataset, SAMPLE ONLY
⚠️ This file is ~832 MB. Do NOT load the full file for profiling. Load only a sample (`nrows=50000`) to inspect structure. The full file stays git-ignored and is only used later, sampled/filtered, if actually needed for a specific KPI.

In [13]:
inpatient_files = [f for f in os.listdir(INPATIENT_DIR) if f.endswith('.csv')]
print("Inpatient Discharges files found:", inpatient_files)

Inpatient Discharges files found: ['Hospital_Inpatient_Discharges__SPARCS_De-Identified___2021_20231012.csv']


In [14]:
inpatient_path = f"{INPATIENT_DIR}/{inpatient_files[0]}"

# SAMPLE only — 50,000 rows — never load the full 832MB file in this notebook
inpatient_sample = pd.read_csv(inpatient_path, nrows=50000)

profile(inpatient_sample, "Inpatient Discharges (SAMPLE — 50,000 rows)")

C:\Users\memyo\AppData\Local\Temp\ipykernel_8496\2556358090.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  inpatient_sample = pd.read_csv(inpatient_path, nrows=50000)



Inpatient Discharges (SAMPLE — 50,000 rows)
Shape: (50000, 33)

Columns: ['Hospital Service Area', 'Hospital County', 'Operating Certificate Number', 'Permanent Facility Id', 'Facility Name', 'Age Group', 'Zip Code - 3 digits', 'Gender', 'Race', 'Ethnicity', 'Length of Stay', 'Type of Admission', 'Patient Disposition', 'Discharge Year', 'CCSR Diagnosis Code', 'CCSR Diagnosis Description', 'CCSR Procedure Code', 'CCSR Procedure Description', 'APR DRG Code', 'APR DRG Description', 'APR MDC Code', 'APR MDC Description', 'APR Severity of Illness Code', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Payment Typology 1', 'Payment Typology 2', 'Payment Typology 3', 'Birth Weight', 'Emergency Department Indicator', 'Total Charges', 'Total Costs']

Dtypes:
 Hospital Service Area                   object
Hospital County                         object
Operating Certificate Number           float64
Permanent Facility Id                  float6

,Hospital Service Area,Hospital County,Operating Certificate Number,Permanent Facility Id,Facility Name,Age Group,Zip Code - 3 digits,Gender,Race,Ethnicity,Length of Stay,Type of Admission,Patient Disposition,Discharge Year,CCSR Diagnosis Code,CCSR Diagnosis Description,CCSR Procedure Code,CCSR Procedure Description,APR DRG Code,APR DRG Description,APR MDC Code,APR MDC Description,APR Severity of Illness Code,APR Severity of Illness Description,APR Risk of Mortality,APR Medical Surgical Description,Payment Typology 1,Payment Typology 2,Payment Typology 3,Birth Weight,Emergency Department Indicator,Total Charges,Total Costs
0,New York City,Bronx,7000006.0,1169.0,Montefiore Medical Center - Henry & Lucy Moses...,70 or Older,104,M,Other Race,Spanish/Hispanic,27,Emergency,Home w/ Home Health Services,2021,INF012,CORONAVIRUS DISEASE 2019 (COVID-19),OTR004,ISOLATION PROCEDURES,137,MAJOR RESPIRATORY INFECTIONS AND INFLAMMATIONS,4,DISEASES AND DISORDERS OF THE RESPIRATORY SYSTEM,3,Major,Extreme,Medical,Medicare,Medicaid,NaN,NaN,Y,"320,922.43","60,241.34"
1,New York City,Bronx,7000006.0,1169.0,Montefiore Medical Center - Henry & Lucy Moses...,50 to 69,104,F,White,Not Span/Hispanic,4,Emergency,Home or Self Care,2021,NVS005,MULTIPLE SCLEROSIS,NaN,NaN,43,"MULTIPLE SCLEROSIS, OTHER DEMYELINATING DISEAS...",1,DISEASES AND DISORDERS OF THE NERVOUS SYSTEM,2,Moderate,Minor,Medical,Private Health Insurance,NaN,NaN,NaN,Y,"61,665.22","9,180.69"
2,New York City,Bronx,7000006.0,1168.0,Montefiore Medical Center-Wakefield Hospital,18 to 29,104,F,Other Race,Spanish/Hispanic,2,Emergency,Home or Self Care,2021,PRG016,PREVIOUS C-SECTION,PGN003,CESAREAN SECTION,540,CESAREAN SECTION WITHOUT STERILIZATION,14,"PREGNANCY, CHILDBIRTH AND THE PUERPERIUM",1,Minor,Minor,Surgical,Medicaid,NaN,NaN,NaN,N,"42,705.34","11,366.5"


## 5. Summary — Dataset Validation Sheet (draft)
This table is the starting point for the mentor-required **Dataset Validation & Mapping Sheet**. Fill in the blanks after reviewing the outputs above.

In [15]:
summary_rows = []

for name, df in hmis_tables.items():
    summary_rows.append({
        'dataset': 'HMIS', 'table': name, 'rows': df.shape[0], 'columns': df.shape[1],
        'missing_pct': round(df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100, 2),
        'duplicates': df.duplicated().sum()
    })

for name, df in [('patients', beds_patients), ('staff', beds_staff),
                  ('staff_schedule', beds_schedule), ('services_weekly', beds_services)]:
    summary_rows.append({
        'dataset': 'Beds Management', 'table': name, 'rows': df.shape[0], 'columns': df.shape[1],
        'missing_pct': round(df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100, 2),
        'duplicates': df.duplicated().sum()
    })

summary_rows.append({
    'dataset': 'Readmission', 'table': readmit_file, 'rows': readmission.shape[0], 'columns': readmission.shape[1],
    'missing_pct': round(readmission.isnull().sum().sum() / (readmission.shape[0]*readmission.shape[1]) * 100, 2),
    'duplicates': readmission.duplicated().sum()
})

summary_rows.append({
    'dataset': 'Inpatient Discharges', 'table': f"{inpatient_files[0]} (sample)",
    'rows': inpatient_sample.shape[0], 'columns': inpatient_sample.shape[1],
    'missing_pct': round(inpatient_sample.isnull().sum().sum() / (inpatient_sample.shape[0]*inpatient_sample.shape[1]) * 100, 2),
    'duplicates': inpatient_sample.duplicated().sum()
})

summary_df = pd.DataFrame(summary_rows)
summary_df

,dataset,table,rows,columns,missing_pct,duplicates
0,HMIS,admission,45000,10,0.00,0
1,HMIS,bed,415,4,0.00,0
2,HMIS,billing,45000,8,0.00,0
3,HMIS,billing_detail,112402,5,11.99,0
4,HMIS,department,11,5,0.00,0
5,HMIS,diagnostic_test,9,5,0.00,0
6,HMIS,disease,20,3,0.00,0
7,HMIS,doctor,98,5,0.00,0
8,HMIS,drug,250,6,0.00,0
9,HMIS,drug_inventory,250,6,0.00,0


In [16]:
os.makedirs('../docs', exist_ok=True)
summary_df.to_csv('../docs/dataset_validation_summary.csv', index=False)
print("Saved to ../docs/dataset_validation_summary.csv")

Saved to ../docs/dataset_validation_summary.csv


## Next steps (do NOT do yet — for planning only)
1. Review the printed columns above for every HMIS table and confirm exact key names (`hospital_id`, `patient_id`, `admission_id`, `department_id`, `ward_id`, `bed_id`).
2. Confirm whether `readmission` dataset shares a `patient_id`/`admission_id` with HMIS, or is standalone (in which case its readmission info can only be used descriptively, not joined row-level).
3. Confirm Inpatient Discharges (SPARCS) column names for LOS/admission/discharge — decide if it's needed at all given HMIS may already cover admissions.
4. Only after all of the above is documented, proceed to building the 4 grain-based tables:
   - `hospital_overview_dataset.csv` (one row = one admission)
   - `patient_flow_dataset.csv` (one row = one movement event)
   - `department_analytics_dataset.csv` (one row = hospital+department+day)
   - `resource_utilization_dataset.csv` (one row = hospital+department+day+resource type)